# AIIMS ECG-PPG Sync Debug Notebook (fixed)

This notebook keeps the **same file paths, sync variable names, and anchor index values** as your original version, but changes the PPG detection logic to avoid non-physiological PTT artifacts.

### Why this version
- The earlier approach used `argmax` in a broad `[0.1, 0.8] s` window after every R-peak.
- That often picks a late wave / motion artifact, producing unrealistically high PTT and floor/ceiling clipping.
- Here, we use:
  1. stable drift correction using relative-time anchor mapping,
  2. robust local peak detection (`find_peaks` + prominence),
  3. physiological gating of PTT.

### Physiological target window used
From common ECG-to-finger-PPG transit timing reported in literature and device studies, practical adult ranges are often around **0.15-0.35 s**; we use a conservative acceptance gate of **0.10-0.45 s**.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import neurokit2 as nk


## 1) Keep the same paths and variable names


In [ ]:
ppg_csv = Path("1/Polar/E90F3223/Fateh Singh_108505253_E90F3223_13ppg.csv")
ecg_csv = Path("1/Holter/06-08-2025.csv")
ppg_csv_device_2 = Path("1/Polar/E9124620/Fateh Singh_108505253_E9124620_14ppg.csv")


def build_device_file_bundle(ppg_path):
    stem = ppg_path.stem
    if not stem.endswith("ppg"):
        raise ValueError(f"Unexpected PPG filename format: {ppg_path.name}")

    prefix = stem[:-3]
    return {
        "ppg": ppg_path,
        "metaData": ppg_path.with_name(f"{prefix}metaData.csv"),
        "accelerometer": ppg_path.with_name(f"{prefix}accelerometer.csv"),
        "gyroscope": ppg_path.with_name(f"{prefix}gyroscope.csv"),
        "magnetometer": ppg_path.with_name(f"{prefix}magnetometer.csv"),
    }


device_1_files = build_device_file_bundle(ppg_csv)
device_2_files = build_device_file_bundle(ppg_csv_device_2)


In [ ]:
fs_ecg = 200

def load_device_streams(device_files, device_label):
    device_data = {
        "label": device_label,
        "paths": device_files,
        "ppg_df": pd.read_csv(device_files["ppg"]),
        "meta_df": pd.read_csv(device_files["metaData"]),
        "sensor_dfs": {},
    }

    for sensor_name in ["accelerometer", "gyroscope", "magnetometer"]:
        sensor_path = device_files[sensor_name]
        if sensor_path.exists():
            device_data["sensor_dfs"][sensor_name] = pd.read_csv(sensor_path)
        else:
            print(f"Warning: {device_label} {sensor_name} file not found at {sensor_path}; skipping sync for this sensor.")

    return device_data


device_1_data = load_device_streams(device_1_files, "Device 1 (E90F3223)")
device_2_data = load_device_streams(device_2_files, "Device 2 (E9124620)")

ecg_df = pd.read_csv(ecg_csv)
ppg_df = device_1_data["ppg_df"]
ppg_df_device_2 = device_2_data["ppg_df"]
meta_df = device_1_data["meta_df"]
meta_df_device_2 = device_2_data["meta_df"]

ecg_signal = ecg_df.iloc[:, 2].to_numpy()
ppg_signal = ppg_df.iloc[:, 1].to_numpy()
ppg_signal_device_2 = ppg_df_device_2.iloc[:, 1].to_numpy()

# derive true sampling rates from timestamps
fs_ppg = 1.0 / np.mean(np.diff(ppg_df["time"].to_numpy()) / 1e9)
fs_ppg_device_2 = 1.0 / np.mean(np.diff(ppg_df_device_2["time"].to_numpy()) / 1e9)

print(f"ECG fs={fs_ecg:.3f} Hz | PPG1 fs={fs_ppg:.3f} Hz | PPG2 fs={fs_ppg_device_2:.3f} Hz")

for device_data in [device_1_data, device_2_data]:
    print(f"Loaded metadata columns for {device_data['label']}:", ", ".join(device_data["meta_df"].columns.tolist()))
    for sensor_name, sensor_df in device_data["sensor_dfs"].items():
        sensor_fs = 1.0 / np.mean(np.diff(sensor_df["time"].to_numpy(np.int64)) / 1e9)
        print(f"{device_data['label']} | {sensor_name.title()} fs={sensor_fs:.3f} Hz | samples={len(sensor_df)}")


## 2) Keep the same sync anchor index values


In [ ]:
# Device 1 (E90F3223) anchors - SAME as original notebook
ppg_idx_1 = 14343639
ppg_idx_2 = 27155637

# ECG anchors - SAME as original notebook
ecg_idx_1 = 16331017
ecg_idx_2 = 30954051

# Device 2 (E9124620) anchors - SAME as original notebook
ppg_idx_1_device_2 = 14403716
ppg_idx_2_device_2 = 27272110


## 3) Signal cleaning and R-peak detection


In [ ]:
ecg_clean = nk.ecg_clean(ecg_signal, sampling_rate=fs_ecg, method="neurokit")
_, ecg_info = nk.ecg_peaks(ecg_clean, sampling_rate=fs_ecg)
rpeaks = ecg_info["ECG_R_Peaks"]

ppg_clean = nk.ppg_clean(ppg_signal, sampling_rate=fs_ppg, method="elgendi")
ppg_clean_device_2 = nk.ppg_clean(ppg_signal_device_2, sampling_rate=fs_ppg_device_2, method="elgendi")

print("R-peaks:", len(rpeaks))


## 4) Stable clock-map fit (relative-time domain)


In [ ]:
def fit_linear_clock_map_from_anchors(ppg_time_ns, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx):
    ppg_anchor_ns = ppg_time_ns[np.asarray(ppg_anchor_idx, dtype=int)]
    ecg_anchor_ns = ecg_time_ns[np.asarray(ecg_anchor_idx, dtype=int)]

    t_ref_ns = int(min(ppg_anchor_ns.min(), ecg_anchor_ns.min()))
    t_ppg_rel = (ppg_anchor_ns - t_ref_ns) / 1e9
    t_ecg_rel = (ecg_anchor_ns - t_ref_ns) / 1e9

    a, b_rel = np.polyfit(t_ppg_rel, t_ecg_rel, deg=1)
    return {
        "a": a,
        "b_rel": b_rel,
        "t_ref_ns": t_ref_ns,
        "ppg_anchor_ns": ppg_anchor_ns,
        "ecg_anchor_ns": ecg_anchor_ns,
    }


def resample_ppg_to_ecg_domain_stable(ppg_time_ns, ppg_clean_sig, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx):
    map_info = fit_linear_clock_map_from_anchors(ppg_time_ns, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx)

    t_ref_ns = map_info["t_ref_ns"]
    a = map_info["a"]
    b_rel = map_info["b_rel"]

    t_ppg_rel = (ppg_time_ns.astype(np.int64) - t_ref_ns) / 1e9
    t_ecg_rel = (ecg_time_ns.astype(np.int64) - t_ref_ns) / 1e9

    # invert map: t_ecg_rel = a * t_ppg_rel + b_rel
    t_ecg_as_ppg_rel = (t_ecg_rel - b_rel) / a
    ppg_resampled = np.interp(t_ecg_as_ppg_rel, t_ppg_rel, ppg_clean_sig)

    return ppg_resampled, map_info


## 5) Robust ECG-guided PPG peak detection (fix)

Key difference vs old code:
- old: choose `argmax` in a very broad window (`0.1-0.8 s`), which can lock onto noise/late waves.
- new: detect candidate peaks with prominence, then choose best candidate with a weighted score.

Candidate score (lower is better):
`score = w_delay * |delay - expected_delay| + w_shape * (1 - normalized_prominence) + w_amp * (1 - normalized_amplitude)`

This keeps the PPG peak physiologically plausible (delay term) while preferring cleaner peaks (shape/amplitude terms).


In [ ]:
def robust_ecg_guided_ppg_peaks(
    rpeak_indices,
    ppg_resampled,
    fs_ecg,
    min_delay=0.10,
    max_delay=0.45,
    expected_delay=0.22,
    min_prom_factor=0.25,
    w_delay=0.70,
    w_shape=0.20,
    w_amp=0.10,
):
    ppg_peak_indices = []
    matched_rpeaks = []

    min_s = int(min_delay * fs_ecg)
    max_s = int(max_delay * fs_ecg)

    for r_idx in rpeak_indices:
        start_idx = r_idx + min_s
        end_idx = r_idx + max_s
        if end_idx >= len(ppg_resampled) or start_idx < 0:
            continue

        window = ppg_resampled[start_idx:end_idx]
        if len(window) < 5:
            continue

        # robust local prominence threshold
        amp = np.nanpercentile(window, 95) - np.nanpercentile(window, 5)
        min_prom = max(1e-6, min_prom_factor * amp)

        cand, props = find_peaks(window, distance=int(0.08 * fs_ecg), prominence=min_prom)
        if len(cand) == 0:
            # fallback: conservative local max only when no peak candidates exist
            cand = np.array([int(np.argmax(window))])
            prom = np.array([0.0], dtype=float)
        else:
            prom = props.get("prominences", np.zeros(len(cand), dtype=float))

        cand_amp = window[cand]
        delays = (cand + min_s) / fs_ecg

        # normalize candidate features for stable weighted scoring
        prom_norm = prom / (np.max(prom) + 1e-8)
        amp_norm = (cand_amp - np.min(cand_amp)) / (np.ptp(cand_amp) + 1e-8)
        delay_norm = np.abs(delays - expected_delay) / (max_delay - min_delay + 1e-8)

        score = w_delay * delay_norm + w_shape * (1.0 - prom_norm) + w_amp * (1.0 - amp_norm)
        best = cand[int(np.argmin(score))]

        ppg_idx = start_idx + int(best)
        ptt = (ppg_idx - r_idx) / fs_ecg
        if min_delay <= ptt <= max_delay:
            ppg_peak_indices.append(ppg_idx)
            matched_rpeaks.append(r_idx)

    return np.asarray(ppg_peak_indices, dtype=int), np.asarray(matched_rpeaks, dtype=int)


def mad_outlier_mask(x, thresh=5.0):
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-9
    z = 0.6745 * (x - med) / mad
    return np.abs(z) <= thresh



## 6) Device 1: before/after drift correction and final PTT


In [ ]:
# BEFORE drift correction (naive interpolation in absolute timestamp domain)
t_ppg_abs = ppg_df["time"].to_numpy(np.int64) / 1e9
t_ecg_abs = ecg_df["time"].to_numpy(np.int64) / 1e9
ppg_abs_to_ecg = np.interp(t_ecg_abs, t_ppg_abs, ppg_clean)

ppg_pk_before, r_before = robust_ecg_guided_ppg_peaks(rpeaks, ppg_abs_to_ecg, fs_ecg)
ptt_before = (ppg_pk_before - r_before) / fs_ecg

# AFTER drift correction
ppg_resampled_device_1_stable, map_device_1 = resample_ppg_to_ecg_domain_stable(
    ppg_df["time"].to_numpy(np.int64),
    ppg_clean,
    ecg_df["time"].to_numpy(np.int64),
    [ppg_idx_1, ppg_idx_2],
    [ecg_idx_1, ecg_idx_2],
)

ppg_pk_after, r_after = robust_ecg_guided_ppg_peaks(rpeaks, ppg_resampled_device_1_stable, fs_ecg)
ptt_after = (ppg_pk_after - r_after) / fs_ecg

# remove residual outliers
mask_after = mad_outlier_mask(ptt_after, thresh=5.0)
ptt_after_clean = ptt_after[mask_after]
beat_time_after = r_after[mask_after] / fs_ecg

offset_device_1_sec = map_device_1["b_rel"]
offset_device_1_ns = offset_device_1_sec * 1e9
print("Device1 map a:", map_device_1["a"], " drift ppm:", (map_device_1["a"] - 1) * 1e6)
print("Device1 offset b_rel (s):", offset_device_1_sec, " | offset (ns):", offset_device_1_ns)
print("PTT before mean±std:", np.mean(ptt_before), np.std(ptt_before), "N=", len(ptt_before))
print("PTT after mean±std :", np.mean(ptt_after_clean), np.std(ptt_after_clean), "N=", len(ptt_after_clean))


In [ ]:
# trend check
coef_before = np.polyfit(r_before / fs_ecg, ptt_before, 1) if len(ptt_before) > 2 else [np.nan, np.nan]
coef_after = np.polyfit(beat_time_after, ptt_after_clean, 1) if len(ptt_after_clean) > 2 else [np.nan, np.nan]

fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax[0].plot(r_before / fs_ecg, ptt_before, '.', ms=1, alpha=0.35, color='tab:green')
if np.isfinite(coef_before[0]):
    x = r_before / fs_ecg
    ax[0].plot(x, coef_before[0] * x + coef_before[1], 'k', lw=2)
ax[0].set_title("Device 1: PTT vs time (before drift correction)")
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("PTT (s)")
ax[0].set_ylim(0.08, 0.8)
ax[0].grid(alpha=0.2)

ax[1].plot(beat_time_after, ptt_after_clean, '.', ms=1, alpha=0.35, color='tab:red')
if np.isfinite(coef_after[0]):
    x = beat_time_after
    ax[1].plot(x, coef_after[0] * x + coef_after[1], 'k', lw=2)
ax[1].set_title("Device 1: PTT vs time (after drift correction)")
ax[1].set_xlabel("Time (s)")
ax[1].set_ylim(0.08, 0.8)
ax[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Slope before: {coef_before[0]:.3e} sec/sec")
print(f"Slope after : {coef_after[0]:.3e} sec/sec")


### 6b) Extra visual diagnostics for PTT shift
Add complementary plots to directly inspect how much drift correction shifts beat-to-beat PTT estimates.

In [ ]:
# extra PTT-shift diagnostics (before vs after)
# IMPORTANT: align by the same matched R-peak indices to avoid index-mismatch bias.
before_map = {int(r): float(ptt) for r, ptt in zip(r_before, ptt_before)}
after_map = {int(r): float(ptt) for r, ptt in zip(r_after, ptt_after)}
common_r = np.array(sorted(set(before_map).intersection(after_map)), dtype=int)

if len(common_r) < 10:
    print("Not enough common beats for reliable shift diagnostics.")
else:
    ptt_before_common = np.array([before_map[int(r)] for r in common_r], dtype=float)
    ptt_after_common = np.array([after_map[int(r)] for r in common_r], dtype=float)
    ptt_shift = ptt_after_common - ptt_before_common
    beat_idx = np.arange(len(common_r))

    # robust rolling medians for clearer trend
    win = max(11, int(0.01 * len(common_r)))
    if win % 2 == 0:
        win += 1
    before_roll = pd.Series(ptt_before_common).rolling(win, center=True, min_periods=max(5, win // 4)).median()
    after_roll = pd.Series(ptt_after_common).rolling(win, center=True, min_periods=max(5, win // 4)).median()

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

    # (1) overlay with rolling medians
    ax[0].plot(beat_idx, ptt_before_common, '.', ms=1, alpha=0.25, color='tab:green', label='Before (common beats)')
    ax[0].plot(beat_idx, ptt_after_common, '.', ms=1, alpha=0.25, color='tab:red', label='After (common beats)')
    ax[0].plot(beat_idx, before_roll, color='darkgreen', lw=2, label=f'Before median (w={win})')
    ax[0].plot(beat_idx, after_roll, color='darkred', lw=2, label=f'After median (w={win})')
    ax[0].set_title('Beat-wise PTT comparison (aligned by R-peak)')
    ax[0].set_xlabel('Common beat index')
    ax[0].set_ylabel('PTT (s)')
    ax[0].set_ylim(0.08, 0.8)
    ax[0].grid(alpha=0.2)
    ax[0].legend(fontsize=8)

    # (2) shift over beats
    ax[1].plot(beat_idx, ptt_shift, '.', ms=1, alpha=0.35, color='tab:purple')
    shift_roll = pd.Series(ptt_shift).rolling(win, center=True, min_periods=max(5, win // 4)).median()
    ax[1].plot(beat_idx, shift_roll, color='purple', lw=2)
    ax[1].axhline(0, color='k', lw=1, ls='--')
    ax[1].set_title('PTT shift = after - before')
    ax[1].set_xlabel('Common beat index')
    ax[1].set_ylabel('Shift (s)')
    ax[1].grid(alpha=0.2)

    # (3) distribution summary
    ax[2].hist(ptt_shift, bins=100, color='tab:purple', alpha=0.75)
    ax[2].axvline(np.median(ptt_shift), color='k', lw=2, ls='--', label=f"median={np.median(ptt_shift):.4f}s")
    ax[2].set_title('Distribution of PTT shift (common beats)')
    ax[2].set_xlabel('Shift (s)')
    ax[2].set_ylabel('Count')
    ax[2].grid(alpha=0.2)
    ax[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"Common beats used: {len(common_r)}")
    print(f"PTT shift median: {np.median(ptt_shift):.6f} s")
    print(f"PTT shift IQR   : {np.percentile(ptt_shift, 75) - np.percentile(ptt_shift, 25):.6f} s")



## 7) Device 2 (same strategy, same anchor variables)


In [ ]:
ppg_resampled_device_2_stable, map_device_2 = resample_ppg_to_ecg_domain_stable(
    ppg_df_device_2["time"].to_numpy(np.int64),
    ppg_clean_device_2,
    ecg_df["time"].to_numpy(np.int64),
    [ppg_idx_1_device_2, ppg_idx_2_device_2],
    [ecg_idx_1, ecg_idx_2],
)

ppg_pk_d2, r_d2 = robust_ecg_guided_ppg_peaks(rpeaks, ppg_resampled_device_2_stable, fs_ecg)
ptt_d2 = (ppg_pk_d2 - r_d2) / fs_ecg
mask_d2 = mad_outlier_mask(ptt_d2, thresh=5.0)
ptt_d2_clean = ptt_d2[mask_d2]

offset_device_2_sec = map_device_2["b_rel"]
offset_device_2_ns = offset_device_2_sec * 1e9
print("Device2 map a:", map_device_2["a"], " drift ppm:", (map_device_2["a"] - 1) * 1e6)
print("Device2 offset b_rel (s):", offset_device_2_sec, " | offset (ns):", offset_device_2_ns)
print("Device2 PTT mean±std:", np.mean(ptt_d2_clean), np.std(ptt_d2_clean), "N=", len(ptt_d2_clean))


## 7) Sync accelerometer, gyroscope, and magnetometer to the ECG clock

Each Polar PPG device has its **own** `accelerometer.csv`, `gyroscope.csv`, `magnetometer.csv`, and `metaData.csv` files in the same device folder. So the sync must be done **per device**:
- Device 1 IMU files use the Device 1 PPG file naming prefix.
- Device 2 IMU files use the Device 2 PPG file naming prefix.

### How the sync works
1. Start from each device's `...ppg.csv` path.
2. Derive the matching `...metaData.csv`, `...accelerometer.csv`, `...gyroscope.csv`, and `...magnetometer.csv` paths from the same filename prefix.
3. Reuse that same device's PPG↔ECG anchor fit `t_ecg_rel = a * t_polar_rel + b_rel` because the IMU streams and PPG stream for a given device share the same Polar clock.
4. For every ECG timestamp, invert the device clock map to find the equivalent Polar-time sample location.
5. Interpolate each numeric sensor channel onto the ECG timestamps.
6. Write synced CSVs with the ECG `time` column so ECG, PPG, ACC, GYRO, and MAGN are aligned on the same timebase for that device.


In [ ]:
def map_timebase_to_source_rel(target_time_ns, map_info):
    t_ref_ns = map_info["t_ref_ns"]
    a = map_info["a"]
    b_rel = map_info["b_rel"]

    target_time_rel = (np.asarray(target_time_ns, dtype=np.int64) - t_ref_ns) / 1e9
    return (target_time_rel - b_rel) / a


def build_synced_output_path(source_path):
    source_path = Path(source_path)
    device_dir = source_path.parent
    parent_dir = device_dir.parent if device_dir.parent != device_dir else device_dir
    synced_dir = parent_dir / f"Synched_{device_dir.name}"
    synced_dir.mkdir(parents=True, exist_ok=True)
    return synced_dir / f"sync{source_path.name}"


def resample_with_known_clock_map(source_time_ns, source_values, ecg_time_ns, map_info):
    t_ref_ns = map_info["t_ref_ns"]
    source_time_rel = (np.asarray(source_time_ns, dtype=np.int64) - t_ref_ns) / 1e9
    ecg_as_source_rel = map_timebase_to_source_rel(ecg_time_ns, map_info)

    source_values = np.asarray(source_values, dtype=np.float32)
    if source_values.ndim == 1:
        return np.interp(ecg_as_source_rel, source_time_rel, source_values).astype(np.float32)

    resampled_columns = [
        np.interp(ecg_as_source_rel, source_time_rel, source_values[:, col_idx]).astype(np.float32)
        for col_idx in range(source_values.shape[1])
    ]
    return np.column_stack(resampled_columns)


def write_synced_signal_file(source_path, ecg_time_ns, synced_columns, file_label, chunk_size=500_000):
    output_path = build_synced_output_path(source_path)
    column_names = list(synced_columns.keys())
    total_rows = len(ecg_time_ns)

    with output_path.open("w", newline="") as f:
        f.write("time")
        for col_name in column_names:
            f.write(f",{col_name}")
        f.write("\n")

        for start_idx in range(0, total_rows, chunk_size):
            end_idx = min(start_idx + chunk_size, total_rows)
            chunk_df = pd.DataFrame({"time": np.asarray(ecg_time_ns[start_idx:end_idx], dtype=np.int64)})
            for col_name, values in synced_columns.items():
                chunk_df[col_name] = np.asarray(values[start_idx:end_idx], dtype=np.float32)
            chunk_df.to_csv(f, index=False, header=False)

    print(f"Saved synced {file_label} to: {output_path}")
    return output_path


def sync_sensor_dataframe_to_ecg(sensor_df, ecg_time_ns, map_info, sensor_name, sensor_prefix, chunk_size=500_000):
    time_col = "time" if "time" in sensor_df.columns else sensor_df.columns[0]
    value_columns = [
        col for col in sensor_df.columns
        if col != time_col and pd.api.types.is_numeric_dtype(sensor_df[col])
    ]

    if not value_columns:
        raise ValueError(f"No numeric value columns found for {sensor_name}.")

    source_time_rel = (sensor_df[time_col].to_numpy(np.int64) - map_info["t_ref_ns"]) / 1e9
    source_value_arrays = {
        col: sensor_df[col].to_numpy(np.float32, copy=False)
        for col in value_columns
    }
    source_path = sensor_df.attrs.get("source_path")
    if source_path is None:
        raise ValueError(f"Missing source_path metadata for {sensor_name}.")

    synced_output_path = build_synced_output_path(source_path)
    synced_column_names = [f"{sensor_prefix}_{col}_resampled" for col in value_columns]
    total_rows = len(ecg_time_ns)

    with synced_output_path.open("w", newline="") as f:
        f.write("time")
        for col_name in synced_column_names:
            f.write(f",{col_name}")
        f.write("\n")

        for start_idx in range(0, total_rows, chunk_size):
            end_idx = min(start_idx + chunk_size, total_rows)
            ecg_chunk = np.asarray(ecg_time_ns[start_idx:end_idx], dtype=np.int64)
            ecg_as_source_rel_chunk = map_timebase_to_source_rel(ecg_chunk, map_info)
            chunk_df = pd.DataFrame({"time": ecg_chunk})
            for source_col, synced_col in zip(value_columns, synced_column_names):
                chunk_df[synced_col] = np.interp(
                    ecg_as_source_rel_chunk,
                    source_time_rel,
                    source_value_arrays[source_col],
                ).astype(np.float32)
            chunk_df.to_csv(f, index=False, header=False)

    print(f"Saved synced {sensor_name} to: {synced_output_path}")
    return synced_output_path


def sync_device_sensor_files(device_data, ecg_time_ns, map_info, output_prefix):
    synced_outputs = {}
    for sensor_name, sensor_df in device_data["sensor_dfs"].items():
        synced_outputs[sensor_name] = sync_sensor_dataframe_to_ecg(
            sensor_df=sensor_df,
            ecg_time_ns=ecg_time_ns,
            map_info=map_info,
            sensor_name=f"{output_prefix} {sensor_name}",
            sensor_prefix=sensor_name,
        )
    return synced_outputs


for device_data in (device_1_data, device_2_data):
    for sensor_name, sensor_df in device_data["sensor_dfs"].items():
        sensor_df.attrs["source_path"] = device_data["paths"][sensor_name]


ecg_time_ns = ecg_df["time"].to_numpy(np.int64)

sync_ppg_csv_device_1 = write_synced_signal_file(
    device_1_files["ppg"],
    ecg_time_ns,
    {"ppg_resampled": ppg_resampled_device_1_stable},
    "PPG for Device 1",
)

sync_ppg_csv_device_2 = write_synced_signal_file(
    device_2_files["ppg"],
    ecg_time_ns,
    {"ppg_resampled": ppg_resampled_device_2_stable},
    "PPG for Device 2",
)

synced_sensor_outputs_device_1 = sync_device_sensor_files(
    device_data=device_1_data,
    ecg_time_ns=ecg_time_ns,
    map_info=map_device_1,
    output_prefix="Device 1",
)

synced_sensor_outputs_device_2 = sync_device_sensor_files(
    device_data=device_2_data,
    ecg_time_ns=ecg_time_ns,
    map_info=map_device_2,
    output_prefix="Device 2",
)

print("Device 1 synced sensor outputs:", synced_sensor_outputs_device_1)
print("Device 2 synced sensor outputs:", synced_sensor_outputs_device_2)


In [ ]:
# Keep both drift metrics in ppm and report before/after clearly.
# 1) Clock-map drift (anchor clock-scale a): ppm = (a - 1) * 1e6
# 2) PTT trend drift (same slope formula for BEFORE/AFTER): ppm = slope(sec/sec) * 1e6

beat_time_before = r_before / fs_ecg
beat_time_d2 = r_d2[mask_d2] / fs_ecg

# Device 2 BEFORE correction baseline: naive interpolation in absolute timestamp domain
# (same strategy already used for Device 1 before-correction baseline).
t_ppg2_abs = ppg_df_device_2["time"].to_numpy(np.int64) / 1e9
t_ecg_abs = ecg_df["time"].to_numpy(np.int64) / 1e9
ppg2_abs_to_ecg = np.interp(t_ecg_abs, t_ppg2_abs, ppg_clean_device_2)
ppg_pk_before_d2, r_before_d2 = robust_ecg_guided_ppg_peaks(rpeaks, ppg2_abs_to_ecg, fs_ecg)
ptt_before_d2 = (ppg_pk_before_d2 - r_before_d2) / fs_ecg
beat_time_before_d2 = r_before_d2 / fs_ecg

coef_before_d1 = np.polyfit(beat_time_before, ptt_before, 1) if len(ptt_before) > 2 else [np.nan, np.nan]
coef_after_d1 = np.polyfit(beat_time_after, ptt_after_clean, 1) if len(ptt_after_clean) > 2 else [np.nan, np.nan]
coef_before_d2 = np.polyfit(beat_time_before_d2, ptt_before_d2, 1) if len(ptt_before_d2) > 2 else [np.nan, np.nan]
coef_after_d2 = np.polyfit(beat_time_d2, ptt_d2_clean, 1) if len(ptt_d2_clean) > 2 else [np.nan, np.nan]

# Clock-map drift BEFORE correction (from anchor mapping)
clock_map_before_ppm_device_1 = (map_device_1["a"] - 1) * 1e6
clock_map_before_ppm_device_2 = (map_device_2["a"] - 1) * 1e6

# Clock-map drift AFTER correction (check in corrected time domain; should be ~0 ppm)
t_ppg1 = ppg_df["time"].to_numpy(np.int64)
t_ppg2 = ppg_df_device_2["time"].to_numpy(np.int64)
t_ecg = ecg_df["time"].to_numpy(np.int64)

t_ppg1_rel_anchor = (t_ppg1[[ppg_idx_1, ppg_idx_2]] - t_ppg1[ppg_idx_1]) / 1e9
t_ppg2_rel_anchor = (t_ppg2[[ppg_idx_1_device_2, ppg_idx_2_device_2]] - t_ppg2[ppg_idx_1_device_2]) / 1e9
t_ecg_rel_anchor = (t_ecg[[ecg_idx_1, ecg_idx_2]] - t_ecg[ecg_idx_1]) / 1e9

t_ppg1_corr_rel_anchor = map_device_1["a"] * t_ppg1_rel_anchor + map_device_1["b_rel"]
t_ppg2_corr_rel_anchor = map_device_2["a"] * t_ppg2_rel_anchor + map_device_2["b_rel"]

a_after_d1 = np.polyfit(t_ppg1_corr_rel_anchor, t_ecg_rel_anchor, 1)[0]
a_after_d2 = np.polyfit(t_ppg2_corr_rel_anchor, t_ecg_rel_anchor, 1)[0]
clock_map_after_ppm_device_1 = (a_after_d1 - 1) * 1e6
clock_map_after_ppm_device_2 = (a_after_d2 - 1) * 1e6

pre_corrected_trend_ppm_device_1 = coef_before_d1[0] * 1e6 if np.isfinite(coef_before_d1[0]) else np.nan
post_corrected_residual_ppm_device_1 = coef_after_d1[0] * 1e6 if np.isfinite(coef_after_d1[0]) else np.nan
pre_corrected_trend_ppm_device_2 = coef_before_d2[0] * 1e6 if np.isfinite(coef_before_d2[0]) else np.nan
post_corrected_residual_ppm_device_2 = coef_after_d2[0] * 1e6 if np.isfinite(coef_after_d2[0]) else np.nan

if np.isfinite(pre_corrected_trend_ppm_device_1) and pre_corrected_trend_ppm_device_1 != 0:
    d1_reduction_pct = 100.0 * (1.0 - abs(post_corrected_residual_ppm_device_1) / abs(pre_corrected_trend_ppm_device_1))
else:
    d1_reduction_pct = np.nan

if np.isfinite(pre_corrected_trend_ppm_device_2) and pre_corrected_trend_ppm_device_2 != 0:
    d2_reduction_pct = 100.0 * (1.0 - abs(post_corrected_residual_ppm_device_2) / abs(pre_corrected_trend_ppm_device_2))
else:
    d2_reduction_pct = np.nan

print("How these are calculated:")
print("  • Clock-map drift ppm = (a - 1) * 1e6, where 'a' is the clock-scale from anchor mapping.")
print("  • PTT trend/residual ppm = slope(PTT vs beat_time in sec/sec) * 1e6 using np.polyfit(..., 1).")
print("  • Drift reduction % = 100 * (1 - |after_ppm|/|before_ppm|).")

print(f"Clock-map drift BEFORE correction (Device 1): {clock_map_before_ppm_device_1:.3f} ppm")
print(f"Clock-map drift AFTER correction  (Device 1): {clock_map_after_ppm_device_1:.3f} ppm")
print(f"Clock-map drift BEFORE correction (Device 2): {clock_map_before_ppm_device_2:.3f} ppm")
print(f"Clock-map drift AFTER correction  (Device 2): {clock_map_after_ppm_device_2:.3f} ppm")
print(f"PTT trend drift BEFORE correction (Device 1, same slope formula): {pre_corrected_trend_ppm_device_1:.3f} ppm")
print(f"PTT residual trend AFTER correction (Device 1, same slope formula): {post_corrected_residual_ppm_device_1:.3f} ppm")
print(f"PTT trend drift BEFORE correction (Device 2, same slope formula): {pre_corrected_trend_ppm_device_2:.3f} ppm")
print(f"PTT residual trend AFTER correction (Device 2, same slope formula): {post_corrected_residual_ppm_device_2:.3f} ppm")
print(f"Device 1 drift reduction vs BEFORE: {d1_reduction_pct:.2f}%")
print(f"Device 2 drift reduction vs BEFORE: {d2_reduction_pct:.2f}%")


## 8) Plot ECG vs PPG before/after drift correction (windowed, Plotly)


In [ ]:
def plot_ecg_ppg_windowed_comparison(
    ecg_time_ns,
    ecg_signal,
    ppg_before,
    ppg_after,
    fs_ecg,
    device_name,
    start_idx=None,
    start_time_sec=None,
    window_sec=30,
    min_ptt_sec=None,
):
    """
    Plot ECG + PPG overlays before and after drift correction in a selectable window.

    Choose either start_idx (ECG sample index) or start_time_sec (seconds from ECG start).
    If min_ptt_sec is provided, show only matched peaks with PTT >= min_ptt_sec.
    Marker labels/hover text include the ECG and PPG sample indices for each matched point.
    """
    ecg_time_sec = (ecg_time_ns - ecg_time_ns[0]) / 1e9

    if start_idx is None:
        if start_time_sec is None:
            start_time_sec = 0
        start_idx = int(start_time_sec * fs_ecg)

    start_idx = max(0, min(int(start_idx), len(ecg_signal) - 1))
    win_samples = max(1, int(window_sec * fs_ecg))
    end_idx = min(len(ecg_signal), start_idx + win_samples)

    x = ecg_time_sec[start_idx:end_idx]
    ecg_win = ecg_signal[start_idx:end_idx]
    ppg_before_win = ppg_before[start_idx:end_idx]
    ppg_after_win = ppg_after[start_idx:end_idx]

    rpeaks_win, _ = nk.ecg_peaks(ecg_signal[start_idx:end_idx], sampling_rate=fs_ecg)
    r_local = np.where(rpeaks_win["ECG_R_Peaks"] == 1)[0].astype(int)
    r_global = start_idx + r_local

    ppg_before_peaks, r_before = robust_ecg_guided_ppg_peaks(r_global, ppg_before, fs_ecg)
    ppg_after_peaks, r_after = robust_ecg_guided_ppg_peaks(r_global, ppg_after, fs_ecg)

    if min_ptt_sec is not None:
        before_ptt = (ppg_before_peaks - r_before) / fs_ecg
        keep_before = before_ptt >= float(min_ptt_sec)
        ppg_before_peaks = ppg_before_peaks[keep_before]
        r_before = r_before[keep_before]

        after_ptt = (ppg_after_peaks - r_after) / fs_ecg
        keep_after = after_ptt >= float(min_ptt_sec)
        ppg_after_peaks = ppg_after_peaks[keep_after]
        r_after = r_after[keep_after]

    def pair_segment_trace(r_idx, ppg_idx, ppg_signal, name, color, yaxis):
        x_seg, y_seg = [], []
        for r_i, p_i in zip(r_idx, ppg_idx):
            x_seg.extend([ecg_time_sec[r_i], ecg_time_sec[p_i], None])
            y_seg.extend([ecg_signal[r_i], ppg_signal[p_i], None])
        return go.Scatter(
            x=x_seg,
            y=y_seg,
            mode='lines',
            name=name,
            line=dict(color=color, width=1, dash='dot'),
            opacity=0.35,
            hoverinfo='skip',
            yaxis=yaxis,
        )

    def matched_marker_traces(r_idx, ppg_idx, ppg_signal):
        pair_ids = np.arange(len(r_idx))
        r_text = [f"ECG idx={idx}" for idx in r_idx]
        ppg_text = [f"PPG idx={idx}" for idx in ppg_idx]
        common_custom = np.column_stack([pair_ids, r_idx, ppg_idx]) if len(pair_ids) else np.empty((0, 3), dtype=int)

        r_trace = go.Scatter(
            x=ecg_time_sec[r_idx],
            y=ecg_signal[r_idx],
            mode='markers+text',
            name='R-peaks',
            text=r_text,
            textposition='top center',
            textfont=dict(size=9, color='crimson'),
            marker=dict(symbol='x', size=8, color='crimson'),
            customdata=common_custom,
            hovertemplate='Pair %{customdata[0]}<br>ECG idx=%{customdata[1]}<br>PPG idx=%{customdata[2]}<br>Time=%{x:.3f} s<br>ECG=%{y:.3f}<extra></extra>',
            yaxis='y1',
        )
        ppg_trace = go.Scatter(
            x=ecg_time_sec[ppg_idx],
            y=ppg_signal[ppg_idx],
            mode='markers+text',
            name='PPG peaks',
            text=ppg_text,
            textposition='bottom center',
            textfont=dict(size=9, color='darkorange'),
            marker=dict(symbol='circle', size=7, color='darkorange'),
            customdata=common_custom,
            hovertemplate='Pair %{customdata[0]}<br>ECG idx=%{customdata[1]}<br>PPG idx=%{customdata[2]}<br>Time=%{x:.3f} s<br>PPG=%{y:.3f}<extra></extra>',
            yaxis='y2',
        )
        return r_trace, ppg_trace

    fig_before = go.Figure()
    fig_before.add_trace(go.Scatter(x=x, y=ecg_win, mode='lines', name='ECG (clean)', yaxis='y1'))
    fig_before.add_trace(go.Scatter(x=x, y=ppg_before_win, mode='lines', name='PPG (before drift correction)', yaxis='y2'))
    fig_before.add_trace(pair_segment_trace(r_before, ppg_before_peaks, ppg_before, 'ECG→PPG match', 'gray', 'y2'))
    r_before_trace, ppg_before_trace = matched_marker_traces(r_before, ppg_before_peaks, ppg_before)
    fig_before.add_trace(r_before_trace)
    fig_before.add_trace(ppg_before_trace)
    fig_before.update_layout(
        title=f"{device_name} | BEFORE drift correction | start_idx={start_idx}, window={window_sec}s",
        xaxis=dict(title='Time from ECG start (s)'),
        yaxis=dict(title='ECG amplitude'),
        yaxis2=dict(title='PPG amplitude', overlaying='y', side='right'),
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    )
    fig_before.show()

    fig_after = go.Figure()
    fig_after.add_trace(go.Scatter(x=x, y=ecg_win, mode='lines', name='ECG (clean)', yaxis='y1'))
    fig_after.add_trace(go.Scatter(x=x, y=ppg_after_win, mode='lines', name='PPG (after drift correction)', yaxis='y2'))
    fig_after.add_trace(pair_segment_trace(r_after, ppg_after_peaks, ppg_after, 'ECG→PPG match', 'gray', 'y2'))
    r_after_trace, ppg_after_trace = matched_marker_traces(r_after, ppg_after_peaks, ppg_after)
    fig_after.add_trace(r_after_trace)
    fig_after.add_trace(ppg_after_trace)
    fig_after.update_layout(
        title=f"{device_name} | AFTER drift correction | start_idx={start_idx}, window={window_sec}s",
        xaxis=dict(title='Time from ECG start (s)'),
        yaxis=dict(title='ECG amplitude'),
        yaxis2=dict(title='PPG amplitude', overlaying='y', side='right'),
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    )
    fig_after.show()


# ---- User-adjustable window controls ----
plot_start_idx = 600000   # ECG index (set None to use plot_start_time_sec)
plot_start_time_sec = None
plot_window_sec = 60
plot_min_ptt_sec = None  # e.g., 0.20 to show matched peaks only from this PTT onward

# Device 1
plot_ecg_ppg_windowed_comparison(
    ecg_time_ns=ecg_df['time'].to_numpy(np.int64),
    ecg_signal=ecg_clean,
    ppg_before=ppg_abs_to_ecg,
    ppg_after=ppg_resampled_device_1_stable,
    fs_ecg=fs_ecg,
    device_name='Device 1 (E90F3223)',
    start_idx=plot_start_idx,
    start_time_sec=plot_start_time_sec,
    window_sec=plot_window_sec,
    min_ptt_sec=plot_min_ptt_sec,
)

# Device 2
plot_ecg_ppg_windowed_comparison(
    ecg_time_ns=ecg_df['time'].to_numpy(np.int64),
    ecg_signal=ecg_clean,
    ppg_before=ppg2_abs_to_ecg,
    ppg_after=ppg_resampled_device_2_stable,
    fs_ecg=fs_ecg,
    device_name='Device 2 (E9124620)',
    start_idx=plot_start_idx,
    start_time_sec=plot_start_time_sec,
    window_sec=plot_window_sec,
    min_ptt_sec=plot_min_ptt_sec,
)



## 9) Quick diagnostics for feasible PTT range


In [ ]:
def ptt_summary(name, ptt):
    q = np.nanpercentile(ptt, [1, 5, 25, 50, 75, 95, 99])
    print(f"\n{name}")
    print("  min/max:", np.min(ptt), np.max(ptt))
    print("  p01,p05,p25,p50,p75,p95,p99:", q)

ptt_summary("Device1 (after)", ptt_after_clean)
ptt_summary("Device2 (after)", ptt_d2_clean)

plt.figure(figsize=(8,4))
plt.hist(ptt_after_clean, bins=120, alpha=0.6, label='Device1')
plt.hist(ptt_d2_clean, bins=120, alpha=0.6, label='Device2')
plt.axvline(0.10, color='k', ls='--', lw=1)
plt.axvline(0.45, color='k', ls='--', lw=1)
plt.xlabel("PTT (s)")
plt.ylabel("Count")
plt.title("PTT distribution after robust detection + drift correction")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


### Interpretation checklist

If there are still issues after this fix, inspect these in order:
1. `map_device_1['a']` and `map_device_2['a']` should be close to 1 (small ppm drift).
2. `Slope after` should be much closer to zero than `Slope before`.
3. PTT distribution should cluster mostly in ~0.15-0.35 s (with some spread).
4. If PTT still piles at bounds 0.10 or 0.45, tighten motion filtering and/or revise expected delay.
